In [ ]:
# ============================================================
# AUTOENCODER TRANSFER LEARNING: MNIST -> EMNIST (PHASED)
# ------------------------------------------------------------

import copy
import random
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split, Subset
from torchvision import transforms
from torchvision.datasets import MNIST, EMNIST

# ============================================================
# 1. REPRODUCIBILITY & DEVICE
# ============================================================

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything()

device = torch.device(
    "cuda" if torch.cuda.is_available() 
    else "mps" if torch.backends.mps.is_available() 
    else "cpu"
)

# ============================================================
# 2. DATA PREPARATION 
# ============================================================

emnist_transform = transforms.Compose([
    lambda img: transforms.functional.rotate(img, -90),
    lambda img: transforms.functional.hflip(img),
    transforms.ToTensor()
])

mnist_transform = transforms.ToTensor()

mnist_full = MNIST(root="data", train=True, transform=mnist_transform, download=True)
mnist_test = MNIST(root="data", train=False, transform=mnist_transform, download=True)
generator=torch.Generator().manual_seed(42)
mnist_train, mnist_val = random_split(mnist_full, [55000, 5000], generator=generator)

fraction = 0.01
emnist_full_train = EMNIST(root="data", split="letters", train=True, transform=emnist_transform, download=True)
emnist_test = EMNIST(root="data", split="letters", train=False, transform=emnist_transform, download=True)

subset_size = int(len(emnist_full_train) * fraction)

g = torch.Generator().manual_seed(42)
indices = torch.randperm(
    len(emnist_full_train),
    generator=g
)[:subset_size]

emnist_subset = Subset(emnist_full_train, indices)

train_size = int(0.8 * subset_size)
val_size = subset_size - train_size

generator = torch.Generator().manual_seed(42)

emnist_train, emnist_val = random_split(
    emnist_subset,
    [train_size, val_size],
    generator=generator
)

batch_size = 128
loader_args = {"batch_size": batch_size, "num_workers": 0}

mnist_train_loader = DataLoader(mnist_train, shuffle=True, **loader_args)
mnist_val_loader   = DataLoader(mnist_val, **loader_args)
mnist_test_loader  = DataLoader(mnist_test, **loader_args)

emnist_train_loader = DataLoader(emnist_train, shuffle=True, **loader_args)
emnist_val_loader   = DataLoader(emnist_val, **loader_args)
emnist_test_loader  = DataLoader(emnist_test, **loader_args)

# ============================================================
# 3. MODEL ARCHITECTURE 
# ============================================================

class Autoencoder(nn.Module):
    def __init__(self, latent_dim=32):
        super().__init__()
        self.latent_dim = latent_dim

        self.encoder = nn.Sequential(
            nn.Conv2d(1, 16, 3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(16, 32, 3, stride=2, padding=1),
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(32 * 7 * 7, latent_dim)
        )

        self.decoder_input = nn.Linear(latent_dim, 32 * 7 * 7)
        
        self.decoder = nn.Sequential(
            nn.Upsample(scale_factor=2, mode='nearest'),
            nn.Conv2d(32, 16, 3, padding=1),
            nn.ReLU(),
            nn.Upsample(scale_factor=2, mode='nearest'),
            nn.Conv2d(16, 1, 3, padding=1),
            nn.Sigmoid()
        )

    def forward(self, x):
        z = self.encoder(x)
        x = self.decoder_input(z)
        x = x.view(-1, 32, 7, 7)
        return self.decoder(x)

# ============================================================
# 4. UTILITIES
# ============================================================

def add_noise(x, noise_factor=0.3):
    noise = torch.randn_like(x) * noise_factor
    return torch.clamp(x + noise, 0., 1.)

class EarlyStopping:
    def __init__(self, patience=5):
        self.patience, self.counter, self.best_loss = patience, 0, np.inf
        self.best_weights = None

    def __call__(self, model, val_loss):
        if val_loss < self.best_loss:
            self.best_loss = val_loss
            self.counter = 0
            self.best_weights = copy.deepcopy(model.state_dict())
            return False
        else:
            self.counter += 1
            return self.counter >= self.patience

    def restore(self, model):
        if self.best_weights:
            model.load_state_dict(self.best_weights)

# ============================================================
# 5. TRAINING ENGINE (With Unfreeze Logic)
# ============================================================

def set_encoder_lr(optimizer, lr):
    optimizer.param_groups[0]["lr"] = lr
    
def run_experiment(
    model,
    train_loader,
    val_loader,
    denoising=False,
    epochs=20,
    encoder_lr=1e-3,
    unfreeze_epoch=None,
    fine_tune_lr=1e-4
):
    
    criterion = nn.MSELoss()
    
    # Encoder starts frozen effectively via lr=0.0
    optimizer = torch.optim.Adam([
        {"params": model.encoder.parameters(), "lr": encoder_lr},
        {"params": model.decoder_input.parameters(), "lr": 1e-3},
        {"params": model.decoder.parameters(), "lr": 1e-3},
    ])
    
    es = EarlyStopping(patience=5)
    history = {"train": [], "val": []}
    model.to(device)

    for epoch in range(epochs):
        
        # --- LOGIC FOR UNFREEZING ---
        if unfreeze_epoch is not None and epoch == unfreeze_epoch:
            print(f"\n>>> Epoch {epoch}: Fine-tuning encoder...")
            set_encoder_lr(optimizer, fine_tune_lr)

        # TRAIN
        model.train()
        t_loss = 0
        for x, _ in train_loader:
            x = x.to(device)
            inp = add_noise(x) if denoising else x
            out = model(inp)
            loss = criterion(out, x)
            optimizer.zero_grad(); loss.backward(); optimizer.step()
            t_loss += loss.item()
        
        # VAL
        model.eval()
        v_loss = 0
        with torch.no_grad():
            for x, _ in val_loader:
                x = x.to(device)
                inp = add_noise(x) if denoising else x
                v_loss += criterion(model(inp), x).item()
        
        t_loss /= len(train_loader); v_loss /= len(val_loader)
        history["train"].append(t_loss); history["val"].append(v_loss)
        # Change your print line to something like this:
        enc_lr = optimizer.param_groups[0]["lr"]

        print(
            f"Epoch {epoch+1:02d} | "
            f"Train: {t_loss:.4f} | "
            f"Val: {v_loss:.4f} | "
            f"Encoder LR: {enc_lr}"
        )
        
        if es(model, v_loss): 
            print("Early stopping triggered.")
            break

    es.restore(model)
    return history

def visualize(model, dataset, denoising=False):
    model.eval()
    x = torch.stack([dataset[i][0] for i in range(8)]).to(device)
    inp = add_noise(x) if denoising else x
    with torch.no_grad(): out = model(inp)
    
    fig, axs = plt.subplots(3, 8, figsize=(12, 5))
    for i in range(8):
        axs[0,i].imshow(x[i][0].cpu(), cmap='gray'); axs[0,i].axis('off')
        axs[1,i].imshow(inp[i][0].cpu(), cmap='gray'); axs[1,i].axis('off')
        axs[2,i].imshow(out[i][0].cpu(), cmap='gray'); axs[2,i].axis('off')
    plt.show()

# ============================================================
# 6. EXECUTION
# ============================================================

# --- STEP 1: MNIST PRE-TRAINING ---
print("\n--- STEP 1: MNIST ---")
mnist_ae = Autoencoder(latent_dim=64)
run_experiment(
    model=mnist_ae,
    train_loader=mnist_train_loader,
    val_loader=mnist_val_loader,
    denoising=False,
    epochs=20,
)
torch.save(mnist_ae.state_dict(), "mnist_base.pt")

# --- STEP 2: TRANSFER WITH PHASED UNFREEZING ---
print("\n--- STEP 2: Transfer Learning (Frozen for 5 epochs, then Full Fine-Tuning) ---")
transfer_ae = Autoencoder(latent_dim=64)
transfer_ae.load_state_dict(torch.load("mnist_base.pt"))

# We pass encoder_lr=0.0 to simulate the frozen phase. 
# At epoch 5, the engine scales it up to fine_tune_lr=1e-4.
transfer_hist = run_experiment(
    model=transfer_ae,
    train_loader=emnist_train_loader,
    val_loader=emnist_val_loader,
    denoising=True,
    epochs=80,
    encoder_lr=0.0,
    unfreeze_epoch=5,
    fine_tune_lr=1e-4,
)
visualize(transfer_ae, emnist_test, denoising=True)

# --- STEP 3: SCRATCH ---
print("\n--- STEP 3: From Scratch ---")
scratch_ae = Autoencoder(latent_dim=64)
scratch_hist = run_experiment(
    model=scratch_ae,
    train_loader=emnist_train_loader,
    val_loader=emnist_val_loader,
    denoising=True,
    epochs=80,
    encoder_lr=1e-3,
)
visualize(scratch_ae, emnist_test, denoising=True)

# --- FINAL COMPARISON ---
plt.plot(transfer_hist["val"], label="Transfer (Phased)")
plt.plot(scratch_hist["val"], label="Scratch")
plt.title("Phased Transfer vs Scratch")
plt.xlabel("Epoch")
plt.ylabel("Validation Loss")
plt.legend()
plt.show()